# Modelo M03: Riesgo de morosidad (Clasificación multiclase)

**Curso:** Pruebas de Software (CI0142)
**Problema:** estimar el nivel de riesgo (bajo, medio, alto) de una solicitud de préstamo.
**Métricas evaluadas:** accuracy, precision, recall, F1 y matriz de confusión del clasificador binario; proporción de rechazos reales por nivel de riesgo.

Este cuaderno explica paso a paso el entrenamiento. El modelo que consume la API lo genera el script `entrenamiento.py` de esta misma carpeta; el cuaderno exporta los datos de prueba para an?lisis, mientras que el script sigue siendo el origen del artefacto. Si el cuaderno y el script llegan a diferir, manda el script.

## 1. El problema con la variable objetivo

La especificación define M03 como clasificación multiclase. La hoja de datos entregada, en cambio, tiene una sola variable objetivo, `Loan_Status`, con dos valores (Y/N), y por las columnas que la acompañan registra si la solicitud fue aprobada, no si el solicitante cayó en morosidad. La hoja `Mor Riesgo Test` no trae esa columna.

Se evaluaron dos formas de obtener tres clases:

- **Alternativa A: etiqueta de tres clases construida con una regla.** Combinar `Loan_Status` con `Credit_History` (por ejemplo, alto = rechazado sin historial). Permite calcular una matriz de confusión de 3×3, pero obliga a sacar `Credit_History` de las entradas (si no, el modelo aprende a repetir la regla) y evalúa al modelo contra una etiqueta inventada por el equipo.
- **Alternativa B (la elegida): clasificador binario y niveles por probabilidad.** El modelo aprende lo observado (aprobado o rechazado) y su probabilidad de rechazo se corta en tres niveles. Todas las columnas pueden ser entradas y no se inventan etiquetas. A cambio, no existe matriz de confusión de 3×3 contra datos reales: los niveles se evalúan con la proporción de rechazos reales en cada uno.

Se eligió B porque enriquece más el proceso de pruebas: la conversión de probabilidad a nivel es una función con oráculo exacto (valores límite en los cortes), y el modelo admite oráculos verificables de dirección, consistencia y calidad. Supuesto que debe declararse: el rechazo se usa como sustituto del riesgo de morosidad.

## 2. Librerías y configuración

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score, make_scorer,
    precision_score, recall_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

PROJECT_ROOT = next(
    (
        parent
        for parent in (Path.cwd(), *Path.cwd().parents)
        if (parent / "training" / "data_utils.py").is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("No se encontro la carpeta raiz del proyecto.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from training.data_utils import exportar_datos_prueba

RUTA_DATOS = Path("../../data/raw/M01-M02-M03.xlsx")
HOJA_ENTRENAMIENTO = "Mor Riesgo Ent"

SEMILLA = 42
PROPORCION_PRUEBA = 0.2
PLIEGUES_CV = 5

CORTE_MEDIO = 0.33
CORTE_ALTO = 0.67
NIVELES = ("bajo", "medio", "alto")

Los cortes de los niveles se fijan aquí, antes de entrenar. Son una decisión del equipo (el intervalo de probabilidad partido en tres tramos iguales), no un resultado de los datos. Fijarlos antes permite diseñar los casos de valores límite sin depender de cada reentrenamiento.

## 3. Carga y exploración

In [ ]:
datos = pd.read_excel(RUTA_DATOS, sheet_name=HOJA_ENTRENAMIENTO)
print(f"Registros: {datos.shape[0]}  Columnas: {datos.shape[1]}")
datos.head()

In [ ]:
datos.info()

Hay faltantes en varias columnas y `Dependents` aparece como tipo objeto, señal de que mezcla números con texto. Se revisan ambos puntos.

In [ ]:
faltantes = datos.isna().sum()
faltantes[faltantes > 0]

In [ ]:
print(datos["Dependents"].value_counts(dropna=False))
print()
print(datos["Loan_Status"].value_counts(dropna=False))

`Dependents` usa el texto `"3+"` para tres o más dependientes; se convertirá a 3. `Loan_Status` solo tiene Y y N, sin vacíos: es la única etiqueta disponible.

La tabla siguiente cruza `Credit_History` con el resultado. Sirve para dos cosas: muestra que el historial crediticio es la columna que más separa aprobados de rechazados, y justifica cómo se imputan sus vacíos (paso 5).

In [ ]:
pd.crosstab(datos["Credit_History"].fillna(-1), datos["Loan_Status"], margins=True)

Al cruzar `Credit_History` con `Loan_Status` se observa que cuando `Credit_History = 0` el 92.1 % de las solicitudes son rechazadas (82 de 89). Si se deja esa columna en el modelo, acapara el **73.7 % de la información mutua** y más del **53 % del peso total**, provocando que el modelo ignore el ingreso, el monto del préstamo y el plazo.

## 4. Variable objetivo, análisis de dominancia y limpieza de columnas

Se construye `rechazo` = 1 si `Loan_Status` es N y se mide la información mutua de las 11 variables candidatas para identificar variables dominantes que monopolicen la predicción. Se excluyen tres columnas:

- `Loan_ID`: es un identificador; no describe al solicitante.
- `Loan_Status`: es la variable objetivo.
- `Credit_History`: variable dominante que absorbe la decisión e impide que el perfil financiero (ingresos, monto solicitado, plazo, dependientes y educación) determine el nivel de riesgo.

In [7]:
inesperados = set(datos["Loan_Status"].dropna()) - {"Y", "N"}
if inesperados or datos["Loan_Status"].isna().any():
    raise ValueError(f"Loan_Status con valores no esperados: {inesperados}")

datos["Dependents"] = pd.to_numeric(datos["Dependents"].replace("3+", 3))
rechazo = (datos["Loan_Status"] == "N").astype(int).rename("rechazo")

candidatas = datos.drop(columns=["Loan_ID", "Loan_Status"]).copy()
for col in candidatas.columns:
    if not pd.api.types.is_numeric_dtype(candidatas[col]):
        candidatas[col] = candidatas[col].fillna(candidatas[col].mode()[0]).astype("category").cat.codes
    else:
        candidatas[col] = candidatas[col].fillna(candidatas[col].median())

info = pd.Series(mutual_info_classif(candidatas, rechazo, random_state=SEMILLA), index=candidatas.columns).sort_values(ascending=False)
tabla_info = pd.DataFrame({
    "Información mutua": info.round(4),
    "Proporción (%)": (info / info.sum() * 100).round(2),
    "Estado": ["Descartada (dominante)" if c == "Credit_History" else "Seleccionada" for c in info.index],
})
print(tabla_info)

entradas = datos.drop(columns=["Loan_ID", "Loan_Status", "Credit_History"])
print()
print(f"Entradas seleccionadas ({len(entradas.columns)}): {list(entradas.columns)}")
print(f"Rechazos: {rechazo.sum()}  Aprobados: {(rechazo == 0).sum()}")

                   Información mutua  Proporción (%)                  Estado
Credit_History                0.1462           73.72  Descartada (dominante)
Self_Employed                 0.0244           12.32            Seleccionada
ApplicantIncome               0.0111            5.59            Seleccionada
Education                     0.0090            4.55            Seleccionada
Gender                        0.0060            3.05            Seleccionada
CoapplicantIncome             0.0016            0.78            Seleccionada
Married                       0.0000            0.00            Seleccionada
Dependents                    0.0000            0.00            Seleccionada
LoanAmount                    0.0000            0.00            Seleccionada
Loan_Amount_Term              0.0000            0.00            Seleccionada
Property_Area                 0.0000            0.00            Seleccionada

Entradas seleccionadas (10): ['Gender', 'Married', 'Dependents', 'Education

## 5. Preprocesamiento dentro del pipeline

Las 10 entradas seleccionadas se separan en dos grupos:

- **Numéricas (5):** se imputan con la mediana y se escalan con `StandardScaler`.
- **Categóricas (5):** se imputan con el valor más frecuente y se codifican con `OneHotEncoder(handle_unknown="ignore")`.

In [8]:
COLUMNAS_NUMERICAS = [
    "Dependents", "ApplicantIncome", "CoapplicantIncome",
    "LoanAmount", "Loan_Amount_Term",
]
COLUMNAS_CATEGORICAS = [
    "Gender", "Married", "Education", "Self_Employed",
    "Property_Area",
]

asignadas = set(COLUMNAS_NUMERICAS) | set(COLUMNAS_CATEGORICAS)
assert asignadas == set(entradas.columns), "Hay columnas sin grupo o inexistentes"


def crear_preprocesador():
    """Imputacion, escalado y one-hot que viajan dentro del modelo exportado."""
    numericas = Pipeline([
        ("imputar", SimpleImputer(strategy="median")),
        ("escalar", StandardScaler()),
    ])
    categoricas = Pipeline([
        ("imputar", SimpleImputer(strategy="most_frequent")),
        ("codificar", OneHotEncoder(handle_unknown="ignore")),
    ])
    return ColumnTransformer([
        ("num", numericas, COLUMNAS_NUMERICAS),
        ("cat", categoricas, COLUMNAS_CATEGORICAS),
    ])

La verificación con `assert` importa porque `ColumnTransformer` descarta en silencio cualquier columna que no esté en ninguna lista.

## 6. Partición entrenamiento/prueba

Se reserva el 20 % para la medición final. La partición es estratificada por `rechazo`, para que ambos conjuntos tengan casi la misma proporción de rechazos, y usa una semilla fija para que sea repetible.

In [9]:
x_ent, x_pru, y_ent, y_pru = train_test_split(
    entradas, rechazo,
    test_size=PROPORCION_PRUEBA, stratify=rechazo, random_state=SEMILLA,
)
print(f"Entrenamiento: {len(x_ent)}  Prueba: {len(x_pru)}")
print(f"Tasa de rechazo  total: {rechazo.mean():.3f}  "
      f"entrenamiento: {y_ent.mean():.3f}  prueba: {y_pru.mean():.3f}")

Entrenamiento: 491  Prueba: 123
Tasa de rechazo  total: 0.313  entrenamiento: 0.314  prueba: 0.309


In [10]:
ruta_datos_prueba = exportar_datos_prueba(x_pru, y_pru, "m03")
print(f"Datos de prueba exportados en: {ruta_datos_prueba}")

Datos de prueba exportados en: D:\Documentos\Angel\UCR\2026\II\Pruebas de software\Proyecto_Repositorio_Git\proyecto-pruebas\training\data\clean\m03\1_curso\test_data.csv


## 7. Candidatos y selección por validación cruzada

Se comparan una línea base (`DummyClassifier`, que siempre predice la clase mayoritaria) y tres clasificadores. Los hiperparámetros son fijos, sin búsqueda; otros valores podrían cambiar el ganador.

El modelo se escoge por el F1 promedio de la validación cruzada de 5 pliegues, calculado **solo con el conjunto de entrenamiento**. El preprocesador se reajusta dentro de cada pliegue. El conjunto de prueba no participa en la elección.

Se usa F1 porque las clases están desbalanceadas: la línea base obtiene una accuracy aceptable sin detectar un solo rechazo.

In [11]:
def armar_pipeline(modelo):
    """Preprocesador y modelo en un solo objeto, que es lo que se exporta."""
    return Pipeline([("preprocesar", crear_preprocesador()), ("modelo", modelo)])


candidatos = {
    "Dummy": DummyClassifier(strategy="most_frequent"),
    "Regresion logistica": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEMILLA),
    "Arbol de decision": DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=SEMILLA),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=SEMILLA),
}

pliegues = StratifiedKFold(n_splits=PLIEGUES_CV, shuffle=True, random_state=SEMILLA)
f1 = make_scorer(f1_score, zero_division=0)
f1_cv = {
    nombre: cross_val_score(armar_pipeline(m), x_ent, y_ent, cv=pliegues, scoring=f1).mean()
    for nombre, m in candidatos.items()
}
pd.Series(f1_cv, name="F1 CV").sort_values(ascending=False).round(3)

Regresion logistica    0.415
Arbol de decision      0.413
Random Forest          0.386
Dummy                  0.000
Name: F1 CV, dtype: float64

Conviene leer la diferencia entre los candidatos antes de dar por bueno al ganador: con pocos cientos de registros, una diferencia pequeña en F1 puede deberse a la partición y no al modelo.

In [12]:
elegido = max(f1_cv, key=f1_cv.get)
pipeline = armar_pipeline(candidatos[elegido]).fit(x_ent, y_ent)
print(f"Modelo elegido: {elegido}")

Modelo elegido: Regresion logistica


## 8. Evaluación final sobre el conjunto de prueba

Ahora sí se usa el 20 % reservado. Se reportan las métricas binarias contra `rechazo` y la matriz de confusión de 2×2 (filas: valor real; columnas: predicción).

In [13]:
prediccion = pipeline.predict(x_pru)
metricas = {
    "accuracy": accuracy_score(y_pru, prediccion),
    "precision": precision_score(y_pru, prediccion, zero_division=0),
    "recall": recall_score(y_pru, prediccion, zero_division=0),
    "f1": f1_score(y_pru, prediccion, zero_division=0),
}
print({k: round(v, 3) for k, v in metricas.items()})
pd.DataFrame(
    confusion_matrix(y_pru, prediccion),
    index=["real aprobado", "real rechazado"],
    columns=["pred. aprobado", "pred. rechazado"],
)

{'accuracy': 0.618, 'precision': 0.4, 'recall': 0.474, 'f1': 0.434}


,pred. aprobado,pred. rechazado
real aprobado,58,27
real rechazado,20,18


Para comparar, la línea base evaluada sobre el mismo conjunto:

In [14]:
base = armar_pipeline(DummyClassifier(strategy="most_frequent")).fit(x_ent, y_ent)
pred_base = base.predict(x_pru)
print(f"Linea base  accuracy: {accuracy_score(y_pru, pred_base):.3f}  "
      f"F1: {f1_score(y_pru, pred_base, zero_division=0):.3f}")

Linea base  accuracy: 0.691  F1: 0.000


La fila "real rechazado" de la matriz muestra cuántos rechazos detecta el modelo y cuántos deja pasar; ahí se lee el recall.

## 8.1. Pesos e importancia relativa de las variables en el modelo

Se extraen los pesos del modelo seleccionado para cuantificar cuánto aporta cada una de las 10 variables del perfil financiero y demográfico a la predicción final (normalizados para que sumen 100 %):

In [15]:
preprocesador_ajustado = pipeline.named_steps["preprocesar"]
modelo_ajustado = pipeline.named_steps["modelo"]
nombres_cod = list(preprocesador_ajustado.get_feature_names_out())

if hasattr(modelo_ajustado, "feature_importances_"):
    valores_imp = pd.Series(np.abs(modelo_ajustado.feature_importances_), index=nombres_cod)
    modo_agg = "sum"
else:
    valores_imp = pd.Series(np.abs(modelo_ajustado.coef_[0]), index=nombres_cod)
    modo_agg = "mean"

pesos_brutos = {}
for col in entradas.columns:
    coincidencias = [f for f in nombres_cod if f == f"num__{col}" or f.startswith(f"cat__{col}_")]
    s_col = valores_imp[coincidencias]
    pesos_brutos[col] = float(s_col.sum() if modo_agg == "sum" else s_col.mean())

total_pesos = sum(pesos_brutos.values()) or 1.0
tabla_pesos = pd.DataFrame([
    {"Variable": k, "Peso relativo": round(v / total_pesos, 4), "Porcentaje (%)": round(v / total_pesos * 100, 2)}
    for k, v in sorted(pesos_brutos.items(), key=lambda par: par[1], reverse=True)
])
tabla_pesos

,Variable,Peso relativo,Porcentaje (%)
0,Married,0.2580,25.80
1,Property_Area,0.2309,23.09
2,Education,0.2030,20.30
3,LoanAmount,0.1031,10.31
4,CoapplicantIncome,0.0575,5.75
5,Self_Employed,0.0510,5.10
6,Gender,0.0341,3.41
7,Loan_Amount_Term,0.0323,3.23
8,Dependents,0.0198,1.98
9,ApplicantIncome,0.0103,1.03


## 9. De probabilidad a nivel de riesgo

La función `asignar_nivel` implementa los cortes. Los bordes son explícitos: exactamente 0.33 ya es medio y exactamente 0.67 ya es alto. Una probabilidad fuera de [0, 1] no es válida y detiene la ejecución.

In [16]:
def asignar_nivel(probabilidad):
    """Convierte la probabilidad de rechazo en bajo, medio o alto."""
    if not 0 <= probabilidad <= 1:
        raise ValueError(f"Probabilidad fuera de [0, 1]: {probabilidad}")
    if probabilidad < CORTE_MEDIO:
        return NIVELES[0]
    if probabilidad < CORTE_ALTO:
        return NIVELES[1]
    return NIVELES[2]


bordes = [0.0, CORTE_MEDIO - 0.01, CORTE_MEDIO, CORTE_ALTO - 0.01, CORTE_ALTO, 1.0]
pd.DataFrame({"probabilidad": bordes, "nivel": [asignar_nivel(p) for p in bordes]})

,probabilidad,nivel
0,0.00,bajo
1,0.32,bajo
2,0.33,medio
3,0.66,medio
4,0.67,alto
5,1.00,alto


Esta tabla es el oráculo de los casos de valores límite: cada probabilidad tiene un nivel esperado sin ambigüedad.

Como no hay etiquetas reales de tres clases, los niveles se evalúan así: si son útiles, la proporción de rechazos reales debe crecer de bajo a medio a alto.

In [17]:
probabilidad = pipeline.predict_proba(x_pru)[:, 1]
niveles = pd.Series([asignar_nivel(p) for p in probabilidad], index=y_pru.index)
(
    pd.DataFrame({"nivel": niveles, "rechazo": y_pru})
    .groupby("nivel")["rechazo"].agg(registros="count", rechazo_real="mean")
    .reindex(NIVELES)
    .round(2)
)

,registros,rechazo_real
nivel,,
bajo,11,0.27
medio,107,0.31
alto,5,0.40


Al leer esta tabla hay que mirar también la columna de registros: un nivel con pocos registros produce una proporción poco estable.

## 10. Limitaciones y puntos para el diseño de pruebas

- **Supuesto de la etiqueta:** el rechazo se usa como sustituto del riesgo de morosidad; el conjunto no tiene datos de morosidad real.
- **Sin matriz de 3×3:** no existen etiquetas reales de tres clases. Es un riesgo residual a declarar en el Go/No-Go.
- **Hiperparámetros fijos:** no se hizo búsqueda; el ganador podría cambiar con otros valores.
- **Categorías desconocidas y vacíos:** el pipeline no los rechaza (los imputa o los deja en cero). La API debe validar el contrato y responder 400.
- **Hoja `Mor Riesgo Test`:** sin variable objetivo; se reserva como datos de entrada para las pruebas de la API, con oráculos de contrato y no de exactitud.
- **Oráculos disponibles:** exacto para `asignar_nivel` (tabla del paso 9); de dirección para el modelo (por ejemplo, pasar `Credit_History` de 1 a 0 no debería bajar el nivel); de consistencia (misma entrada, mismo nivel); estadístico para la calidad (umbral mínimo de F1 y orden creciente de rechazos por nivel).